# Chapter 2 - Primer on Probability Modeling

The notebook is a code companion to chapter 2 of the book [Causal AI](https://www.manning.com/books/causal-ai) by [Robert Osazuwa Ness](https://www.linkedin.com/in/osazuwa/).

<a href="https://colab.research.google.com/github/altdeep/causalAI/blob/master/book/chapter%202/Chapter_2_Primer_on_Probability_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code was written with pgmpy version 0.1.24 and pyro version 1.8.6. Install from within a Python environment using:

In [ ]:
!pip install pgmpy==0.1.24
!pip install pyro-ppl==1.8.6

pgmpy depends on pandas. The version of pandas used was 1.5.3. Check your version with:

In [ ]:
import pandas as pd
print(pd.__version__)


## Listing 2.1: Implementing a discrete distribution table in pgmpy


In [ ]:
import pgmpy
from pgmpy.factors.discrete import DiscreteFactor
dist = DiscreteFactor(
    variables=["X"],    #A
    cardinality=[3],    #B
    values=[.45, .30, .25],    #C
    state_names= {'X': ['1', '2', '3']}    #D
)

#A A list of the names of the variables in the factor
#B The cardinality (number of possible outcomes of each variable in the factor
#C The values each variable in the factor can take
#D Dictionary where the key is the variable name and the value is a list of the names of that variable’s outcomes

print(dist)

“phi(X)” is the probability value assigned to each outcome of X.  One thing to note is that these phi values don’t need to sum to one.  For example, I can multiple each probability value by one hundred as follows.

In [ ]:
dist = DiscreteFactor(
    variables=["X"],
    cardinality=[3],
    values=[45, 30, 25],
    state_names= {'X': ['1', '2', '3']}
)

print(dist)

pgmpy relaxes the constraint to sum to one is because the relaxation is useful in some algorithms.  We can always normalize to obtain proper probability values.

In [ ]:
# Normalize takes phi values for each outcome and divides them by their sum to get a probability.
dist.normalize()

print(dist)

## Listing 2.2 Modeling a joint distribution in pgmpy

In [ ]:
joint = DiscreteFactor(
    variables=['X', 'Y'],   #A
    cardinality=[3, 2],    #B
    values=[.25, .20, .20, .10, .15, .10],   #C
    state_names= {
        'X': ['1', '2', '3'],    #C
        'Y': ['0', '1']    #C
    }
)

#A Now we have two variables instead of one.
#B X has 3 outcomes, Y has 2.
#C You can look at the printed output to see how the values are ordered of values.
#D Now there are two variables, so we name the outcomes for both variables.

print(joint)

The marginalize method will accomplish summing over the specified variables for us.


In [ ]:
print(joint.marginalize(variables=['Y'], inplace=False))
print(joint.marginalize(variables=['X'], inplace=False))

We can use a division operator on two factors to calculate conditional probabilities. Here, P(X, Y)/P(X) = P(Y|X).

In [ ]:
print(joint / dist)

## Listing 2.3 Canonical parameters in Pyro

However, the more direct way in pgmpy to specify a conditional probability distribution table is with the TabularCPD class:

In [ ]:
from pgmpy.factors.discrete.CPD import TabularCPD
PYgivenX = TabularCPD(
    variable='Y',    #A
    variable_card=2,    #B
    values=[
        [.25/.45, .20/.30, .15/.25],    #C
        [.20/.45, .10/.30, .10/.25],    #C
    ],
    evidence=['X'],
    evidence_card=[3],
    state_names = {
        'X': ['1', '2', '3'],
        'Y': ['0', '1']
    })

#A Conditional distribution has one variable instead of DiscreteFactor’s list of variables.
#B variable_card is the cardinality of Y
#C Elements of the list correspond to outcomes for Y. Elements of each list correspond to elements of X.

print(PYgivenX)

### Canonical parameters in Pyro

In [ ]:
import torch
from pyro.distributions import Bernoulli, Categorical, Gamma, Normal

# The categorical distribution takes a list of probability values, each value corresponding to an outcome.
print(Categorical(probs=torch.tensor([.45, .30, .25])))
print(Normal(loc=0.0, scale=1.0))
print(Bernoulli(probs=0.4))
print(Gamma(concentration=1.0, rate=2.0))

bern = Bernoulli(0.4)
lprob = bern.log_prob(torch.tensor(1.0))

import math
print(math.exp(lprob))

## Listing 2.4: Simulating from DiscreteFactor in pgmpy and Pyro

In [ ]:
from pgmpy.factors.discrete import DiscreteFactor
dist = DiscreteFactor(
    variables=["X"],
    cardinality=[3],
    values=[.45, .30, .25],
    state_names= {'X': ['1', '2', '3']}
)

#n is the number of instances you wish to generate
dist.sample(n=3)

In [ ]:
joint = DiscreteFactor(
    variables=['X', 'Y'],
    cardinality=[3, 2],
    values=[.25, .20, .20, .10, .15, .10],
    state_names= {
        'X': ['1', '2', '3'],
        'Y': ['0', '1']
    }
)

joint.sample(n=3)

Canonical distributions in pyro also use a method with sample method.

In [ ]:
import torch
from pyro.distributions import Categorical
# Generate 1 sample
one_sample = Categorical(probs=torch.tensor([.45, .30, .25])).sample()
print(one_sample)
# Generate 10 samples
samples = Categorical(probs=torch.tensor([.45, .30, .25])).sample(sample_shape=torch.Size([10]))
print(samples)

## Listing 2.5: Creating a random process in pgmpy and pyro.

In [ ]:
from pgmpy.factors.discrete.CPD import TabularCPD
from pgmpy.models import BayesianNetwork
from pgmpy.sampling import BayesianModelSampling

# Values is the probability value assigned to each outcome. The
PZ = TabularCPD(
    variable='Z',
    variable_card=2,
    values=[[.65], [.35]],
    state_names = {
        'Z': ['0', '1']
    })

# The probability values map to the outcomes specified in the state_names argument.
PXgivenZ = TabularCPD(
    variable='X',
    variable_card=2,
    values=[
        [.8, .6],
        [.2, .4],
    ],
    evidence=['Z'],
    evidence_card=[2],
    state_names = {
        'X': ['0', '1'],
        'Z': ['0', '1']
    })

PYgivenX = TabularCPD(
    variable='Y',
    variable_card=3,
    values=[
        [.1, .8],
        [.2, .1],
        [.7, .1],
    ],
    evidence=['X'],
    evidence_card=[2],
    state_names = {
        'Y': ['1', '2', '3'],
        'X': ['0', '1']
    })

model = BayesianNetwork([('Z', 'X'), ('X', 'Y')])
model.add_cpds(PZ, PXgivenZ, PYgivenX)

generator = BayesianModelSampling(model)
generator.forward_sample(size=3)

## Listing 2.6 Working with combinations of canonical distributions in Pyro

In [ ]:
import torch
from pyro.distributions import Bernoulli, Poisson, Gamma

# Represent P(Z) with a Gamma distribution, and sample z.
z = Gamma(7.5, 1.0).sample()    #A
# Represent P(X|Z=z) with a Poisson distribution with location parameter z, and sample x.
x = Poisson(z).sample()    #B
# Represent P(Y|X=x) with a Bernoulli distribution. The probability parameter is a function of y.
y = Bernoulli(x / (5+x)).sample()    #C

print(z, x, y)

## Listing 2.7: Random processes with nuanced control flow in Pyro

In [ ]:
import torch
from pyro.distributions import Bernoulli, Poisson, Gamma
z = Gamma(7.5, 1.0).sample()
x = Poisson(z).sample()
# y is calculated as a sum of random coin flips.
# y is generated from P(Y|X=x) because the number of flips depends on x.
y = torch.tensor(0.0)
for i in range(int(x)):
    y += Bernoulli(.5).sample()
print(z, x, y)

## Listing 2.8: Using functions for random processes and pyro.sample

In the code below, we generate multiple samples and then use the `mean` method to calculate an average of those samples.

In [ ]:
import torch
import pyro
def random_process():
    z = pyro.sample("z", Gamma(7.5, 1.0))
    x = pyro.sample("x", Poisson(z))
    y = torch.tensor(0.0)
    for i in range(int(x)):
        y += pyro.sample(f"y{i}", Bernoulli(.5))    #A
    return y

generated_samples = generator.forward_sample(size=100)
generated_samples['Y'].apply(int).mean()

In [ ]:
generated_samples_ = torch.stack([random_process() for _ in range(100)])
generated_samples_.mean()

In [ ]:
torch.square(generated_samples_).mean()

## Listing 2.9: Monte Carlo estimation in Pyro

In [ ]:
import torch
import pyro
from pyro.distributions import Bernoulli, Gamma, Poisson
# This new version of random_process now returns both z and y.
def random_process():
    z = pyro.sample("z", Gamma(7.5, 1.0))
    x = pyro.sample("x", Poisson(z))
    y = torch.tensor(0.0)
    for i in range(int(x)):
        y += pyro.sample(f"{i}", Bernoulli(.5))
    return z, y

# Generate 1000 instances of z and y using a list comprehension.
generated_samples = [random_process() for _ in range(1000)]
# Turn the individual z tensors into a single tensor,
# then calculate the Monte Carlo estimate via the mean method.
z_mean = torch.stack([z for z, _ in generated_samples]).mean()

print(z_mean)

z_given_y = torch.stack([z for z, y in generated_samples if y == 3])

print(z_given_y.mean())

In [ ]:
generator.forward_sample(size=10)

## Listing 2.10: Generating IID samples in Pyro

In [ ]:
import pyro
from pyro.distributions import Bernoulli, Poisson, Gamma

def model():
    z = pyro.sample("z", Gamma(7.5, 1.0))
    x = pyro.sample("x", Poisson(z))
    # pyro.plate is a context manager for generating conditional independent
    # samples. This instance of pyro.plate will generate 10 IID samples.
    with pyro.plate('IID', 10):
        # Calling pyro.sample to generates a single outcome y, where y is a
        # tensor of 10 IID samples.
        y = pyro.sample("y", Bernoulli(x / (5+x)))
    return y

model()

## Listing 2.11: An example of a data generating process in code form

In [ ]:
def true_dgp(jenny_inclination, brian_inclination, window_strength):    #A
    jenny_throws_rock = jenny_inclination > 0.5    #B
    brian_throws_rock = brian_inclination > 0.5    #B
    if jenny_throws_rock and brian_throws_rock:    #C
        strength_of_impact = 0.8    #C
    elif jenny_throws_rock or brian_throws_rock:    #D
        strength_of_impact = 0.6    #D
    else:    #E
        strength_of_impact = 0.0    #E
    window_breaks = window_strength < strength_of_impact    #F
    return jenny_throws_rock, brian_throws_rock, window_breaks

#A Input variables reflect Jenny and Brian’s inclination to throw and the window strength.
#B Jenny and Brian throw the rock if so inclined.
#C If both Jenny and Brian throw the rock, the total strength of the impact is .8.
#D If either Jenny or Brian throws the rock, the total strength of the impact is .6.
#E Otherwise, no one throws and the strength of impact is 0.
#F If the strength of impact is greater than the strength of the window, the window breaks.

initials = [
    (0.6, 0.31, 0.83),
    (0.48, 0.53, 0.33),
    (0.66, 0.63, 0.75),
    (0.65, 0.66, 0.8),
    (0.48, 0.16, 0.27)
]

data_points = []
# Now we pass each tuple in the initials list as an input to the true_dgp function, generate a data point, and add this to the data_points list.
for jenny_inclination, brian_inclination, window_strength in initials:
    data_points.append(
        true_dgp(
            jenny_inclination, brian_inclination, window_strength
        )
    )

print(data_points)